# Train Models for Quora Question Pairs

This notebook trains two models:
1. A simple model using CountVectorizer + Logistic Regression.
2. An improved model using SentenceBERT embeddings + Logistic Regression.

In [1]:
import pandas as pd
import sklearn
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
from utils import cast_list_as_strings, get_features_from_df, get_sbert_features

/home/jcalatay/miniconda3/envs/quora_test_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data

In [2]:
data_path = "../nlp_deliv1_materials/quora_data.csv"
quora_df = pd.read_csv(data_path)

## 2. Split Data
Using the specific split required in the project guide.

In [3]:
A_df, test_df = train_test_split(quora_df, test_size=0.05, random_state=123)
train_df, val_df = train_test_split(A_df, test_size=0.05, random_state=123)

print('train_df.shape=', train_df.shape)
print('val_df.shape=', val_df.shape)
print('test_df.shape=', test_df.shape)

# Cache splits
train_df.to_csv("train_df.csv", index=False)
val_df.to_csv("val_df.csv", index=False)
test_df.to_csv("test_df.csv", index=False)

train_df.shape= (291897, 6)
val_df.shape= (15363, 6)
test_df.shape= (16172, 6)


## 3. Fit CountVectorizer

In [4]:
q1_train = cast_list_as_strings(list(train_df["question1"]))
q2_train = cast_list_as_strings(list(train_df["question2"]))
all_questions = q1_train + q2_train

count_vectorizer = CountVectorizer(ngram_range=(1,1))
count_vectorizer.fit(all_questions)

CountVectorizer()

## 4. Train Simple Model

In [5]:
X_tr_simple = get_features_from_df(train_df, count_vectorizer)
y_train = train_df["is_duplicate"].values

logistic_simple = LogisticRegression(solver="liblinear", random_state=123)
logistic_simple.fit(X_tr_simple, y_train)

LogisticRegression(random_state=123, solver='liblinear')

## 5. Train Improved Model

In [5]:
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
X_tr_advanced = get_sbert_features(train_df, sbert_model)
logistic_advanced = LogisticRegression(solver="liblinear", random_state=123, max_iter=500)
logistic_advanced.fit(X_tr_advanced, y_train)

KeyboardInterrupt: 

## 6. Save Models

In [ ]:
models_dir = "models"
if not os.path.exists(models_dir):
    os.makedirs(models_dir)

joblib.dump(logistic_simple, os.path.join(models_dir, "logistic_simple.joblib"))
joblib.dump(logistic_advanced, os.path.join(models_dir, "logistic_advanced.joblib"))
joblib.dump(count_vectorizer, os.path.join(models_dir, "count_vectorizer.joblib"))
print("Models saved.")